# ChatGPT AI Agent 도입 전후 리뷰 분석: Full Pipeline

이 노트북은 Google Play Store의 ChatGPT 원시 리뷰에서 시작하여
전처리, VADER 감성분석, 감성별 통합 LDA, 하위 LDA, 대표 문서 추출,
시각화까지 모두 새로 계산하는 포트폴리오용 재현 파이프라인이다.

기존 `NOUN_LDA.ipynb`에서 실험한 여러 방법 중 최종 연구 설계를 반영했다.
기존 `final_vader.csv`, LDA 모델 또는 결과 CSV는 입력으로 사용하지 않는다.

**연구 질문**

> AI Agent 도입 전후 ChatGPT 앱 사용자의 긍정·부정 인식과 주요 담론의
> 비중은 어떻게 변화했는가?

**분석 기준**

- Agent 도입일: 2025-07-17
- Before: 도입일 이전
- After: 도입 다음 날부터
- 도입 당일 리뷰는 전환일 불확실성을 고려해 제외
- 전후 별도 LDA가 아니라 감성별 통합 LDA를 학습하여 동일한 토픽 공간에서 비교

## 실행 안내

공개용 기본값은 `RUN_MODE = "sample"`, `USE_CACHE = False`다. 따라서 최신 원시
누적 CSV의 일부를 읽어 같은 분석 파이프라인을 빠르게 점검한다.

빠른 코드 점검에는 `RUN_MODE = "sample"`을 사용한다. Sample 모드도 기존
분석 결과를 읽지 않고 원시 리뷰의 일부를 이용해 같은 파이프라인을 실행한다.

전체 데이터는 200만 건 이상이므로 언어 감지와 NLP 전처리에 상당한 시간이
필요하다. 캐시는 이 노트북이 직접 생성한 중간 산출물만 재사용한다.

In [ ]:
# 최초 실행 환경 예시
# %pip install pandas numpy matplotlib seaborn tqdm joblib langdetect nltk spacy gensim
# !python -m spacy download en_core_web_sm

from __future__ import annotations

import json
import random
import re
import time
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        options = [
            candidate,
            candidate / "projects" / "ai-agent-user-perception",
        ]
        for option in options:
            if (option / "notebooks").exists() and (option / "data").exists():
                return option
    raise FileNotFoundError(
        "Project root not found. Run this notebook inside the project repository."
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "generated"
CACHE_DIR = OUTPUT_DIR / "cache"
TABLE_DIR = OUTPUT_DIR / "tables"
MODEL_DIR = OUTPUT_DIR / "models"
FIGURE_DIR = OUTPUT_DIR / "figures"
REPRESENTATIVE_DIR = OUTPUT_DIR / "representative_docs"

for directory in [
    OUTPUT_DIR,
    CACHE_DIR,
    TABLE_DIR,
    MODEL_DIR,
    FIGURE_DIR,
    REPRESENTATIVE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

RUN_MODE = "sample"         # 공개용 점검은 "sample", 전체 재현은 "full"
SAMPLE_SIZE = 20_000
USE_CACHE = False           # True일 때만 이 노트북이 만든 캐시 사용
AUTO_DOWNLOAD_NLTK = True
N_JOBS = -1

AGENT_DATE = pd.Timestamp("2025-07-17")
SENTIMENT_THRESHOLDS = {"positive": 0.20, "negative": -0.20}

MAIN_K_RANGE = range(5, 11)
SUB_K_RANGE = range(3, 9)
COHERENCE_SAMPLE_PER_PERIOD = 25_000
SEARCH_PASSES = 6
FINAL_PASSES = 15
ITERATIONS = 100
NO_BELOW = 5
NO_ABOVE = 0.50
MIN_TOKENS = 3
PARENT_TOPIC_PROB_THRESHOLD = 0.20
MIN_SUB_DOCS_PER_PERIOD = 30

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RUN_MODE    : {RUN_MODE}")
print(f"USE_CACHE  : {USE_CACHE}")

## 0. 기존 연구 그래프 스타일

In [ ]:
class TC:
    NAVY = "#002855"
    LAVENDER = "#B5A9D0"
    GOLD = "#C8972A"
    WHITE = "#FFFFFF"
    TEXT_D = "#1A1A2E"
    TEXT_M = "#5B6E8A"
    GRID = "#E8E8E8"
    INC = "#1B4F72"
    DEC = "#7B241C"
    NEUTRAL = "#566573"
    BEFORE = "#2E75B6"
    AFTER = "#CA6F1E"
    POSITIVE = "#2E75B6"
    NEUTRAL_SENTIMENT = "#B5A9D0"
    NEGATIVE = "#7B241C"


font_candidates = [
    PROJECT_ROOT / "assets" / "fonts",
]
for font_dir in font_candidates:
    if font_dir.exists():
        for font_path in sorted(font_dir.glob("Helvetica*.ttf")):
            fm.fontManager.addfont(str(font_path))

mpl.rcParams.update(
    {
        "font.family": ["Helvetica", "Malgun Gothic", "DejaVu Sans"],
        "axes.unicode_minus": False,
        "figure.facecolor": TC.WHITE,
        "axes.facecolor": TC.WHITE,
        "savefig.facecolor": TC.WHITE,
        "savefig.dpi": 300,
    }
)
sns.set_theme(style="whitegrid")


def style_axis(ax, xlabel: str = "", ylabel: str = "") -> None:
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_color(TC.GRID)
    ax.tick_params(colors=TC.TEXT_M, labelsize=9)
    ax.grid(axis="y", color=TC.GRID, linewidth=0.6)
    ax.grid(axis="x", visible=False)
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=10, color=TC.TEXT_M)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=10, color=TC.TEXT_M)


def save_show(fig, filename: str) -> None:
    fig.tight_layout()
    fig.savefig(
        FIGURE_DIR / filename,
        dpi=300,
        bbox_inches="tight",
        facecolor=TC.WHITE,
    )
    plt.show()
    plt.close(fig)

## 1. 원시 리뷰 로드

`GPT_reviews_YYYYMMDD.csv` 중 날짜가 가장 최신인 파일을 누적 원시 파일로
선택한다. 분석에 필요하지 않은 사용자 이름은 제거하고, 날짜·평점·리뷰를
기준으로 중복을 제거한다.

In [ ]:
RAW_FILE_PATTERN = re.compile(
    r"^GPT_reviews_(\d{8})\.csv$",
    re.IGNORECASE,
)


def select_latest_raw_file(data_dir: Path) -> Path:
    candidates = []
    for path in data_dir.glob("GPT_reviews_*.csv"):
        match = RAW_FILE_PATTERN.match(path.name)
        if match:
            candidates.append((match.group(1), path))
    if not candidates:
        raise FileNotFoundError("GPT_reviews_YYYYMMDD.csv 파일이 없습니다.")
    return max(candidates, key=lambda item: item[0])[1]


def load_raw_reviews(path: Path, run_mode: str) -> pd.DataFrame:
    usecols = lambda column: column in {
        "name", "score", "date", "review"
    }
    if run_mode == "sample":
        target_per_period = max(1, SAMPLE_SIZE // 2)
        reservoirs = {"before": [], "after": []}
        for chunk_number, chunk in enumerate(
            pd.read_csv(path, usecols=usecols, chunksize=100_000)
        ):
            chunk["date"] = pd.to_datetime(
                chunk["date"],
                errors="coerce",
            )
            chunk = chunk.dropna(subset=["date"])
            chunk["sample_period"] = np.select(
                [
                    chunk["date"].lt(AGENT_DATE),
                    chunk["date"].ge(
                        AGENT_DATE + pd.Timedelta(days=1)
                    ),
                ],
                ["before", "after"],
                default="excluded",
            )
            for period in ["before", "after"]:
                group = chunk.loc[
                    chunk["sample_period"].eq(period)
                ].drop(columns=["sample_period"])
                if group.empty:
                    continue
                candidate = pd.concat(
                    [*reservoirs[period], group],
                    ignore_index=True,
                )
                if len(candidate) > target_per_period:
                    candidate = candidate.sample(
                        n=target_per_period,
                        random_state=SEED + chunk_number,
                    )
                reservoirs[period] = [candidate]
        frame = pd.concat(
            [
                *reservoirs["before"],
                *reservoirs["after"],
            ],
            ignore_index=True,
        )
    else:
        frame = pd.read_csv(path, usecols=usecols)
    required = {"score", "date", "review"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"원시 데이터 필수 컬럼 누락: {sorted(missing)}")

    frame = frame.dropna(subset=["score", "date", "review"]).copy()
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    frame["review"] = frame["review"].astype(str).str.strip()
    frame = frame.dropna(subset=["date"])
    frame = frame.loc[frame["review"].ne("")]
    frame = frame.drop_duplicates(
        subset=["date", "score", "review"],
        keep="last",
    ).reset_index(drop=True)

    frame["review_id"] = (
        pd.util.hash_pandas_object(
            frame[["date", "score", "review"]],
            index=False,
        )
        .astype("uint64")
        .astype(str)
    )
    return frame.drop(columns=["name"], errors="ignore")


raw_path = select_latest_raw_file(DATA_DIR)
raw_df = load_raw_reviews(raw_path, RUN_MODE)

print(f"Raw file : {raw_path.name}")
print(f"Rows     : {len(raw_df):,}")
print(f"Period   : {raw_df['date'].min()} ~ {raw_df['date'].max()}")
display(raw_df[["review_id", "score", "date", "review"]].head())

## 2. 언어 감지와 영어 리뷰 선별

기존 분석의 영어 필터 기준을 한 번의 언어 감지 단계로 통합했다.

- langdetect 영어 확률 ≥ 0.90
- 라틴 문자 비율 ≥ 0.80
- NLTK 영어 어휘 비율 ≥ 0.50

기존 노트북처럼 동일 텍스트에 언어 감지를 여러 번 수행하지 않고,
언어·확률·문자 비율·어휘 비율을 한 번 계산해 재사용한다.

In [ ]:
import nltk

NLTK_RESOURCES = {
    "stopwords": "corpora/stopwords",
    "wordnet": "corpora/wordnet",
    "omw-1.4": "corpora/omw-1.4",
    "words": "corpora/words",
    "averaged_perceptron_tagger_eng": (
        "taggers/averaged_perceptron_tagger_eng"
    ),
    "vader_lexicon": "sentiment/vader_lexicon.zip",
}


def ensure_nltk_resources() -> None:
    missing = []
    for package, resource_path in NLTK_RESOURCES.items():
        try:
            nltk.data.find(resource_path)
        except LookupError:
            missing.append(package)

    if missing and AUTO_DOWNLOAD_NLTK:
        print(f"Downloading NLTK resources: {missing}")
        for package in missing:
            nltk.download(package, quiet=True)
    elif missing:
        raise LookupError(
            "NLTK 리소스가 없습니다: "
            + ", ".join(missing)
            + ". AUTO_DOWNLOAD_NLTK=True로 실행하세요."
        )


ensure_nltk_resources()

In [ ]:
from langdetect import DetectorFactory, detect_langs
from langdetect.lang_detect_exception import LangDetectException
from nltk.corpus import words as nltk_words

DetectorFactory.seed = SEED
ENGLISH_VOCAB = {word.lower() for word in nltk_words.words()}


def latin_ratio(text: str) -> float:
    value = str(text)
    if not value.strip():
        return 0.0
    non_latin = re.sub(r"[A-Za-z0-9\s.,!?()\-]", "", value)
    return 1.0 - len(non_latin) / len(value)


def vocab_ratio(text: str) -> float:
    tokens = re.findall(r"[a-z]+", str(text).lower())
    if not tokens:
        return 0.0
    return sum(token in ENGLISH_VOCAB for token in tokens) / len(tokens)


def language_features(text: str) -> tuple[str, float, float, float]:
    language = "unknown"
    probability = 0.0
    try:
        predictions = detect_langs(str(text))
        if predictions:
            best = predictions[0]
            language = best.lang
            probability = float(best.prob)
    except LangDetectException:
        pass
    return (
        language,
        probability,
        latin_ratio(text),
        vocab_ratio(text),
    )


def detect_english_reviews(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    from joblib import Parallel, delayed
    from tqdm.auto import tqdm

    texts = frame["review"].tolist()
    if N_JOBS == 1:
        features = [
            language_features(text)
            for text in tqdm(texts, desc="Language detection")
        ]
    else:
        features = Parallel(
            n_jobs=N_JOBS,
            backend="loky",
            batch_size=500,
        )(
            delayed(language_features)(text)
            for text in tqdm(texts, desc="Language detection dispatch")
        )

    result = frame.copy()
    feature_frame = pd.DataFrame(
        features,
        columns=["lang", "lang_prob", "latin_ratio", "vocab_ratio"],
        index=result.index,
    )
    result = pd.concat([result, feature_frame], axis=1)

    english_mask = (
        result["lang"].eq("en")
        & result["lang_prob"].ge(0.90)
        & result["latin_ratio"].ge(0.80)
        & result["vocab_ratio"].ge(0.50)
    )
    english = result.loc[english_mask].reset_index(drop=True)
    removed = result.loc[~english_mask].copy()
    removed["remove_reason"] = "NON_ENGLISH"
    return english, removed

In [ ]:
CACHE_ANALYSIS = CACHE_DIR / "analysis_reviews.pkl"
CACHE_AUDIT = TABLE_DIR / "preprocessing_audit.csv"

preprocessing_audit = [
    {"stage": "raw_unique", "rows": len(raw_df)}
]

## 3. 텍스트 정규화와 품질 필터

In [ ]:
from nltk import RegexpTokenizer, WordNetLemmatizer
from nltk.corpus import stopwords, wordnet
from tqdm.auto import tqdm

TOKENIZER = RegexpTokenizer(r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)*")
LEMMATIZER = WordNetLemmatizer()
STOP_WORDS = set(stopwords.words("english")) - {
    "no", "nor", "not", "never"
}
STOP_WORDS |= {"app", "apps", "ai"}
MODEL_STOPWORDS = {"gpt", "chatgpt", "openai"}
VERSION_RE = re.compile(r"(?<=\d)\.(?=\d)")
CONTRACTIONS = [
    (re.compile(r"\bcan't\b", re.IGNORECASE), "can not"),
    (re.compile(r"\bwon't\b", re.IGNORECASE), "will not"),
    (re.compile(r"n't\b", re.IGNORECASE), " not"),
]


def wordnet_pos(tag: str):
    return {
        "J": wordnet.ADJ,
        "V": wordnet.VERB,
        "R": wordnet.ADV,
    }.get(tag[0], wordnet.NOUN)


def normalize_text(text: str) -> str:
    value = VERSION_RE.sub("-", str(text)).lower()
    for pattern, replacement in CONTRACTIONS:
        value = pattern.sub(replacement, value)
    return value


def tokenize_for_lemmatization(text: str) -> list[str]:
    tokens = TOKENIZER.tokenize(normalize_text(text))
    stop = STOP_WORDS | MODEL_STOPWORDS
    return [
        token
        for token in tokens
        if len(token) > 1
        and token not in stop
        and not token.isdigit()
    ]


def lemmatize_reviews(
    texts: Iterable[str],
    batch_size: int = 25_000,
) -> tuple[list[str], list[str]]:
    texts = list(texts)
    normalized = [normalize_text(text) for text in texts]
    token_lists = [
        tokenize_for_lemmatization(text) for text in texts
    ]
    cleaned = []

    for start in tqdm(
        range(0, len(token_lists), batch_size),
        desc="NLTK lemmatization",
    ):
        batch = token_lists[start:start + batch_size]
        tagged_sentences = nltk.pos_tag_sents(batch)
        for sentence in tagged_sentences:
            lemmas = [
                LEMMATIZER.lemmatize(word, wordnet_pos(pos))
                for word, pos in sentence
            ]
            cleaned.append(" ".join(lemmas))
    return normalized, cleaned

In [ ]:
MEANINGLESS_RE = re.compile(r"([A-Za-z0-9])\1{3,}")
SYSTEM_PATTERNS = [
    re.compile(r"image generation task was cancelled", re.IGNORECASE),
    re.compile(r"no image was generated", re.IGNORECASE),
]
HINDI_MARKERS = {
    "bahut", "accha", "achcha", "acha", "bohot", "mera", "meri",
    "mere", "hai", "hain", "tha", "thi", "aur", "bhi", "nahi",
    "nahin", "kar", "karo", "karna", "raha", "rahi", "yeh", "woh",
    "wo", "kya", "kyun", "kaise", "zabardast", "kamaal", "namaste",
    "shukriya", "achi", "bura", "bekar", "bakwas", "sundar",
}

PROMPT_LABEL_RE = re.compile(r"\bprompt\s*:", re.IGNORECASE)
PROMPT_IMPERATIVE_RE = re.compile(
    r"^\s*(create|generate|make|edit|convert|replace|remove|enhance|"
    r"retouch|transform|design|draw|render|produce)\b",
    re.IGNORECASE,
)
PROMPT_VERB_RE = re.compile(
    r"\b(create|generate|make|edit|convert|replace|remove|enhance|"
    r"retouch|transform|design|draw|render|produce)\b",
    re.IGNORECASE,
)
PROMPT_STYLE_RE = re.compile(
    r"\b(portrait|cinematic|realistic|ultra[\s-]?realistic|8k|4k|"
    r"studio[\s]?lighting|ghibli|anime[\s]?style|photorealistic|"
    r"hyperrealistic|dslr[\s-]?portrait|editorial[\s-]?portrait|"
    r"dramaticlight)\b",
    re.IGNORECASE,
)
PROMPT_REFERENCE_RE = re.compile(
    r"(uploaded face|reference image|same face|keep the same|"
    r"keep this exact face|based on the image|face reference)",
    re.IGNORECASE,
)
REVIEW_SIGNAL_RE = re.compile(
    r"\b(good|bad|issue|problem|error|subscription|login|crash|bug|"
    r"amazing|love|hate|worst|best|stars|recommend)\b",
    re.IGNORECASE,
)


def review_ttr(text: str) -> float:
    tokens = str(text).lower().split()
    return len(set(tokens)) / len(tokens) if tokens else 0.0


def ngram_repeat_ratio(text: str, n: int = 3) -> float:
    tokens = str(text).lower().split()
    if len(tokens) < n * 2:
        return 0.0
    ngrams = [
        tuple(tokens[index:index + n])
        for index in range(len(tokens) - n + 1)
    ]
    return 1.0 - len(set(ngrams)) / len(ngrams)


def is_hard_prompt(text: str) -> bool:
    value = str(text)
    if PROMPT_LABEL_RE.search(value):
        return True
    imperative = bool(PROMPT_IMPERATIVE_RE.match(value))
    verbs = min(len(PROMPT_VERB_RE.findall(value)), 3)
    styles = min(len(PROMPT_STYLE_RE.findall(value)), 3)
    references = min(len(PROMPT_REFERENCE_RE.findall(value)), 2) * 2
    review_signals = min(len(REVIEW_SIGNAL_RE.findall(value)), 4)
    score = 2 * int(imperative) + verbs + styles + references
    if score >= 6 and review_signals == 0:
        return True
    return (
        imperative
        and (styles >= 1 or references >= 1)
        and review_signals == 0
    )


def quality_remove_reason(review: str, clean_review: str) -> str:
    if len(str(clean_review).split()) < MIN_TOKENS:
        return "TOO_SHORT"
    if review_ttr(review) < 0.25:
        return "LOW_TTR"
    if MEANINGLESS_RE.search(str(review)):
        return "MEANINGLESS"
    if any(pattern.search(str(review)) for pattern in SYSTEM_PATTERNS):
        return "SYSTEM_MESSAGE"
    if ngram_repeat_ratio(review) >= 0.50:
        return "NGRAM_REPEAT"
    if len(set(str(review).lower().split()) & HINDI_MARKERS) >= 2:
        return "ROMANIZED_HINDI"
    if is_hard_prompt(review):
        return "HARD_PROMPT"
    return ""

## 4. spaCy 명사 추출과 최종 필터

In [ ]:
DOMAIN_STOPWORDS = {
    "gpt", "chatgpt", "chat", "app", "application",
    "ai", "openai", "google",
}
FINAL_NOUN_STOPWORDS = {
    "aap", "one", "love", "fun", "info", "stuff", "datum", "guy",
}


def noun_stats(noun_text: str) -> tuple[int, float]:
    tokens = str(noun_text).lower().split()
    if not tokens:
        return 0, 1.0
    top_count = Counter(tokens).most_common(1)[0][1]
    return len(set(tokens)), top_count / len(tokens)


def extract_nouns(
    texts: Iterable[str],
    batch_size: int = 512,
) -> list[str]:
    import spacy
    from tqdm.auto import tqdm

    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    documents = nlp.pipe(
        (str(text) for text in texts),
        batch_size=batch_size,
    )
    output = []
    for document in tqdm(documents, desc="spaCy noun extraction"):
        nouns = [
            token.lemma_.lower()
            for token in document
            if token.pos_ in {"NOUN", "PROPN"}
            and token.is_alpha
            and len(token.lemma_) >= 3
            and token.lemma_.lower() not in DOMAIN_STOPWORDS
        ]
        nouns = [
            noun for noun in nouns
            if noun not in FINAL_NOUN_STOPWORDS
        ]
        output.append(" ".join(nouns))
    return output

In [ ]:
def assign_period(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    after_start = AGENT_DATE + pd.Timedelta(days=1)
    result["period"] = np.select(
        [
            result["date"].lt(AGENT_DATE),
            result["date"].ge(after_start),
        ],
        ["before", "after"],
        default="excluded",
    )
    return result.loc[result["period"].ne("excluded")].reset_index(drop=True)


def run_preprocessing(
    raw: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    audit = [{"stage": "raw_unique", "rows": len(raw)}]

    english, removed_language = detect_english_reviews(raw)
    audit.append({"stage": "english_strict", "rows": len(english)})

    english["review_norm"], english["clean_review"] = lemmatize_reviews(
        english["review"].tolist()
    )
    english["n_tokens"] = english["clean_review"].str.split().str.len()

    from tqdm.auto import tqdm
    tqdm.pandas(desc="Quality filtering")
    english["remove_reason"] = english.progress_apply(
        lambda row: quality_remove_reason(
            row["review"],
            row["clean_review"],
        ),
        axis=1,
    )
    removed_quality = english.loc[
        english["remove_reason"].ne("")
    ].copy()
    kept = english.loc[
        english["remove_reason"].eq("")
    ].drop(columns=["remove_reason"]).reset_index(drop=True)
    audit.append({"stage": "quality_filtered", "rows": len(kept)})

    kept["clean_review_noun"] = extract_nouns(kept["review_norm"])
    stats = kept["clean_review_noun"].map(noun_stats)
    kept["noun_unique_count"] = stats.map(lambda value: value[0])
    kept["noun_max_share"] = stats.map(lambda value: value[1])

    noun_remove_mask = (
        kept["noun_unique_count"].lt(2)
        | kept["noun_max_share"].gt(0.85)
        | kept["clean_review_noun"].str.strip().eq("")
        | kept["review"].map(is_hard_prompt)
    )
    removed_nouns = kept.loc[noun_remove_mask].copy()
    removed_nouns["remove_reason"] = "NOUN_OR_PROMPT_FILTER"
    kept = kept.loc[~noun_remove_mask].reset_index(drop=True)
    audit.append({"stage": "noun_filtered", "rows": len(kept)})

    kept = assign_period(kept)
    audit.append({"stage": "period_assigned", "rows": len(kept)})

    removed = pd.concat(
        [removed_language, removed_quality, removed_nouns],
        ignore_index=True,
        sort=False,
    )
    audit_frame = pd.DataFrame(audit)
    return kept, removed, audit_frame

## 5. VADER 감성분석

In [ ]:
def sentiment_label(compound: float) -> str:
    if compound >= SENTIMENT_THRESHOLDS["positive"]:
        return "positive"
    if compound <= SENTIMENT_THRESHOLDS["negative"]:
        return "negative"
    return "neutral"


def add_vader_sentiment(frame: pd.DataFrame) -> pd.DataFrame:
    from nltk.sentiment import SentimentIntensityAnalyzer
    from tqdm.auto import tqdm

    analyzer = SentimentIntensityAnalyzer()
    tqdm.pandas(desc="VADER sentiment")

    result = frame.copy()
    result["compound"] = result["review"].progress_apply(
        lambda text: analyzer.polarity_scores(str(text))["compound"]
    )
    result["sentiment"] = result["compound"].map(sentiment_label)
    return result

In [ ]:
if USE_CACHE and CACHE_ANALYSIS.exists():
    analysis_df = pd.read_pickle(CACHE_ANALYSIS)
    preprocessing_audit = pd.read_csv(CACHE_AUDIT)
    print("이 노트북이 생성한 전처리 캐시를 사용했습니다.")
else:
    analysis_df, removed_df, preprocessing_audit = run_preprocessing(
        raw_df
    )
    analysis_df = add_vader_sentiment(analysis_df)
    analysis_df.to_pickle(CACHE_ANALYSIS)
    preprocessing_audit.to_csv(CACHE_AUDIT, index=False)
    removed_df.drop(columns=["name"], errors="ignore").to_csv(
        TABLE_DIR / "removed_reviews_log.csv.gz",
        index=False,
        compression="gzip",
    )

required_columns = {
    "review_id", "score", "date", "review", "period",
    "clean_review_noun", "compound", "sentiment",
}
missing = required_columns - set(analysis_df.columns)
if missing:
    raise ValueError(f"분석 데이터 필수 컬럼 누락: {sorted(missing)}")

analysis_df = analysis_df.reset_index(drop=True)
print(f"Final analysis rows: {len(analysis_df):,}")
display(preprocessing_audit)

## 6. 전처리 및 감성분석 시각화

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
plot_data = preprocessing_audit.copy()
bars = ax.bar(
    plot_data["stage"],
    plot_data["rows"],
    color=[TC.NAVY, TC.LAVENDER, TC.GOLD, TC.BEFORE, TC.AFTER][
        :len(plot_data)
    ],
    alpha=0.88,
)
for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{int(bar.get_height()):,}",
        ha="center",
        va="bottom",
        fontsize=9,
        color=TC.TEXT_M,
    )
ax.set_title(
    "Review Count by Preprocessing Stage",
    fontsize=13,
    fontweight="bold",
    color=TC.TEXT_D,
    pad=12,
)
ax.tick_params(axis="x", rotation=20)
style_axis(ax, ylabel="Reviews")
save_show(fig, "01_preprocessing_flow.png")

In [ ]:
sentiment_summary = (
    analysis_df.groupby(["period", "sentiment"])
    .size()
    .rename("count")
    .reset_index()
)
sentiment_summary["share"] = sentiment_summary.groupby(
    "period"
)["count"].transform(lambda values: values / values.sum())
sentiment_summary.to_csv(
    TABLE_DIR / "sentiment_summary.csv",
    index=False,
)

sentiment_order = ["positive", "neutral", "negative"]
sentiment_colors = {
    "positive": TC.POSITIVE,
    "neutral": TC.NEUTRAL_SENTIMENT,
    "negative": TC.NEGATIVE,
}

fig, ax = plt.subplots(figsize=(8, 5))
pivot = (
    sentiment_summary.pivot(
        index="period",
        columns="sentiment",
        values="share",
    )
    .reindex(["before", "after"])
    .fillna(0)
)
bottom = np.zeros(len(pivot))
for sentiment in sentiment_order:
    values = pivot.get(sentiment, pd.Series(0, index=pivot.index))
    ax.bar(
        pivot.index,
        values,
        bottom=bottom,
        color=sentiment_colors[sentiment],
        label=sentiment.title(),
        alpha=0.88,
    )
    bottom += values.to_numpy()

ax.set_title(
    "Sentiment Distribution Before vs. After",
    fontsize=13,
    fontweight="bold",
    color=TC.TEXT_D,
    pad=12,
)
ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
ax.legend(frameon=False, ncol=3, loc="upper center")
style_axis(ax, ylabel="Share")
save_show(fig, "02_sentiment_distribution.png")
display(sentiment_summary)

In [ ]:
monthly = (
    analysis_df.assign(
        month=analysis_df["date"].dt.to_period("M").dt.to_timestamp()
    )
    .groupby("month")
    .agg(
        review_count=("review_id", "size"),
        negative_share=(
            "sentiment",
            lambda values: values.eq("negative").mean(),
        ),
        mean_compound=("compound", "mean"),
    )
    .reset_index()
)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(
    monthly["month"],
    monthly["review_count"],
    color=TC.NAVY,
    linewidth=2,
)
axes[1].plot(
    monthly["month"],
    monthly["negative_share"],
    color=TC.DEC,
    linewidth=2,
)
for axis in axes:
    axis.axvline(
        AGENT_DATE,
        color=TC.GOLD,
        linestyle="--",
        linewidth=1.5,
        label="Agent release",
    )
    axis.legend(frameon=False)
    style_axis(axis)

axes[0].set_title(
    "Monthly Review Volume",
    fontsize=12,
    fontweight="bold",
    color=TC.TEXT_D,
)
axes[1].set_title(
    "Monthly Negative Review Share",
    fontsize=12,
    fontweight="bold",
    color=TC.TEXT_D,
)
axes[1].yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
save_show(fig, "03_monthly_sentiment_trend.png")

## 7. 감성별 통합 LDA

각 감성에서 Before와 After 문서를 합쳐 하나의 LDA를 학습한다.
k 후보별 C_v Coherence를 계산하고 최고값의 k를 선택한다.

k 탐색은 실행 시간을 줄이면서 기간 편향을 방지하기 위해 Before와 After에서
동일한 수의 문서를 표본 추출한다. 최종 LDA는 선택된 k로 해당 감성의 전체
문서를 학습한다.

In [ ]:
from gensim import corpora
from gensim.models import CoherenceModel, LdaModel


@dataclass
class LDAResult:
    name: str
    model: LdaModel
    dictionary: corpora.Dictionary
    documents: pd.DataFrame
    tokens: list[list[str]]
    corpus: list
    selection: pd.DataFrame
    topic_words: pd.DataFrame
    doc_topics: pd.DataFrame
    comparison: pd.DataFrame
    k: int


def tokenize_documents(
    frame: pd.DataFrame,
    column: str = "clean_review_noun",
    min_tokens: int = 2,
) -> tuple[list[list[str]], pd.DataFrame]:
    token_series = frame[column].fillna("").astype(str).str.split()
    valid = token_series.str.len().ge(min_tokens)
    documents = frame.loc[valid].copy().reset_index(drop=True)
    return token_series.loc[valid].tolist(), documents


def build_dictionary_corpus(tokens: list[list[str]]):
    dictionary = corpora.Dictionary(tokens)
    dictionary.filter_extremes(
        no_below=NO_BELOW,
        no_above=NO_ABOVE,
    )
    corpus = [dictionary.doc2bow(document) for document in tokens]
    if len(dictionary) == 0:
        raise ValueError("LDA Dictionary가 비어 있습니다.")
    return dictionary, corpus


def balanced_tuning_sample(
    documents: pd.DataFrame,
    tokens: list[list[str]],
) -> list[list[str]]:
    token_frame = pd.DataFrame(
        {
            "period": documents["period"].to_numpy(),
            "tokens": tokens,
        }
    )
    sampled = []
    for period in ["before", "after"]:
        group = token_frame.loc[token_frame["period"].eq(period)]
        n = min(COHERENCE_SAMPLE_PER_PERIOD, len(group))
        if n:
            sampled.extend(
                group.sample(n=n, random_state=SEED)["tokens"].tolist()
            )
    return sampled

In [ ]:
def select_topic_count(
    documents: pd.DataFrame,
    tokens: list[list[str]],
    k_range: Iterable[int],
    name: str,
) -> tuple[pd.DataFrame, int]:
    tuning_tokens = balanced_tuning_sample(documents, tokens)
    dictionary, corpus = build_dictionary_corpus(tuning_tokens)

    rows = []
    for k in k_range:
        model = LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=k,
            random_state=SEED,
            passes=SEARCH_PASSES,
            iterations=ITERATIONS,
            alpha="auto",
            eta="auto",
        )
        coherence = CoherenceModel(
            model=model,
            texts=tuning_tokens,
            dictionary=dictionary,
            coherence="c_v",
        ).get_coherence()
        rows.append(
            {
                "name": name,
                "k": k,
                "coherence": coherence,
                "log_perplexity": model.log_perplexity(corpus),
            }
        )
        print(f"[{name}] k={k}: C_v={coherence:.4f}")

    selection = pd.DataFrame(rows)
    selected_k = int(
        selection.loc[selection["coherence"].idxmax(), "k"]
    )
    return selection, selected_k


def plot_coherence(
    selection: pd.DataFrame,
    selected_k: int,
    title: str,
    filename: str,
) -> None:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(
        selection["k"],
        selection["coherence"],
        color=TC.NAVY,
        linewidth=2,
        marker="o",
        markersize=6,
    )
    selected = selection.loc[selection["k"].eq(selected_k)].iloc[0]
    ax.scatter(
        [selected_k],
        [selected["coherence"]],
        color=TC.GOLD,
        s=130,
        zorder=4,
        label=(
            f"Selected k={selected_k} "
            f"(C_v={selected['coherence']:.4f})"
        ),
    )
    ax.axvline(
        selected_k,
        color=TC.LAVENDER,
        linestyle="--",
        linewidth=1.2,
    )
    ax.set_xticks(selection["k"])
    ax.set_title(
        title,
        fontsize=13,
        fontweight="bold",
        color=TC.TEXT_D,
        pad=12,
    )
    ax.legend(frameon=False)
    style_axis(
        ax,
        xlabel="Number of Topics (k)",
        ylabel="C_v Coherence Score",
    )
    save_show(fig, filename)

In [ ]:
def extract_topic_words(model: LdaModel, topn: int = 20) -> pd.DataFrame:
    rows = []
    for topic_id in range(model.num_topics):
        for rank, (word, probability) in enumerate(
            model.show_topic(topic_id, topn=topn),
            start=1,
        ):
            rows.append(
                {
                    "topic_num": topic_id + 1,
                    "rank": rank,
                    "word": word,
                    "probability": probability,
                }
            )
    return pd.DataFrame(rows)


def infer_document_topics(
    model: LdaModel,
    corpus: list,
    documents: pd.DataFrame,
) -> pd.DataFrame:
    rows = []
    for document_index, bow in enumerate(corpus):
        distribution = model.get_document_topics(
            bow,
            minimum_probability=0.0,
        )
        probabilities = np.zeros(model.num_topics)
        for topic_id, probability in distribution:
            probabilities[topic_id] = probability

        row = {
            "document_index": document_index,
            "review_id": documents.iloc[document_index]["review_id"],
            "period": documents.iloc[document_index]["period"],
            "dominant_topic": int(probabilities.argmax() + 1),
            "dominant_probability": float(probabilities.max()),
        }
        row.update(
            {
                f"topic_{index + 1}": probability
                for index, probability in enumerate(probabilities)
            }
        )
        rows.append(row)
    return pd.DataFrame(rows)


def compare_topic_proportions(
    doc_topics: pd.DataFrame,
    k: int,
) -> pd.DataFrame:
    topic_columns = [f"topic_{index + 1}" for index in range(k)]
    means = (
        doc_topics.groupby("period")[topic_columns]
        .mean()
        .T
        .rename_axis("topic")
        .reset_index()
    )
    for period in ["before", "after"]:
        if period not in means:
            means[period] = np.nan
    means["delta"] = means["after"] - means["before"]
    means["abs_delta"] = means["delta"].abs()
    means["topic_num"] = (
        means["topic"].str.extract(r"(\d+)")[0].astype(int)
    )
    return means.sort_values("topic_num").reset_index(drop=True)


def train_lda(
    frame: pd.DataFrame,
    name: str,
    k_range: Iterable[int],
    output_prefix: str,
) -> LDAResult:
    tokens, documents = tokenize_documents(frame)
    selection, selected_k = select_topic_count(
        documents,
        tokens,
        k_range,
        name,
    )
    plot_coherence(
        selection,
        selected_k,
        f"{name} LDA: Coherence vs. k",
        f"{output_prefix}_coherence.png",
    )

    dictionary, corpus = build_dictionary_corpus(tokens)
    model = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=selected_k,
        random_state=SEED,
        passes=FINAL_PASSES,
        iterations=ITERATIONS,
        alpha="auto",
        eta="auto",
    )
    topic_words = extract_topic_words(model)
    doc_topics = infer_document_topics(model, corpus, documents)
    comparison = compare_topic_proportions(
        doc_topics,
        selected_k,
    )

    model.save(str(MODEL_DIR / f"{output_prefix}_k{selected_k}.gensim"))
    selection.to_csv(
        TABLE_DIR / f"{output_prefix}_coherence.csv",
        index=False,
    )
    topic_words.to_csv(
        TABLE_DIR / f"{output_prefix}_topic_words.csv",
        index=False,
    )
    doc_topics.to_csv(
        TABLE_DIR / f"{output_prefix}_doc_topics.csv.gz",
        index=False,
        compression="gzip",
    )
    comparison.to_csv(
        TABLE_DIR / f"{output_prefix}_comparison.csv",
        index=False,
    )

    return LDAResult(
        name,
        model,
        dictionary,
        documents,
        tokens,
        corpus,
        selection,
        topic_words,
        doc_topics,
        comparison,
        selected_k,
    )

In [ ]:
main_results = {}
for sentiment, short_name in [
    ("positive", "pos"),
    ("negative", "neg"),
]:
    sentiment_data = (
        analysis_df.loc[analysis_df["sentiment"].eq(sentiment)]
        .reset_index(drop=True)
    )
    print(
        f"\n{sentiment.upper()} LDA input: "
        f"{len(sentiment_data):,} documents"
    )
    main_results[sentiment] = train_lda(
        sentiment_data,
        name=sentiment.upper(),
        k_range=MAIN_K_RANGE,
        output_prefix=f"main_{short_name}",
    )

## 8. 상위 LDA 결과 시각화

In [ ]:
def topic_label(topic_words: pd.DataFrame, topic_num: int, n: int = 4) -> str:
    words = (
        topic_words.loc[topic_words["topic_num"].eq(topic_num)]
        .nsmallest(n, "rank")["word"]
        .tolist()
    )
    return f"T{topic_num}  " + " · ".join(words)


def plot_topic_comparison(
    result: LDAResult,
    prefix: str,
    delta_threshold: float = 0.005,
) -> None:
    comparison = result.comparison.copy()
    comparison["label"] = comparison["topic_num"].map(
        lambda topic_num: topic_label(
            result.topic_words,
            topic_num,
        )
    )
    comparison = comparison.sort_values("topic_num", ascending=False)
    y = np.arange(len(comparison))
    height = 0.36

    fig, ax = plt.subplots(
        figsize=(10, max(4.5, len(comparison) * 0.65))
    )
    before_bars = ax.barh(
        y - height / 2,
        comparison["before"],
        height,
        color=TC.BEFORE,
        alpha=0.85,
        label="Before",
    )
    after_bars = ax.barh(
        y + height / 2,
        comparison["after"],
        height,
        color=TC.AFTER,
        alpha=0.85,
        label="After",
    )
    for bar in [*before_bars, *after_bars]:
        ax.text(
            bar.get_width() + 0.002,
            bar.get_y() + bar.get_height() / 2,
            f"{bar.get_width():.3f}",
            va="center",
            fontsize=8,
            color=TC.TEXT_M,
        )
    ax.set_yticks(y)
    ax.set_yticklabels(comparison["label"])
    ax.set_title(
        f"{result.name} LDA: Topic Proportion Before vs. After",
        fontsize=13,
        fontweight="bold",
        color=TC.TEXT_D,
        pad=12,
    )
    ax.legend(frameon=False)
    style_axis(ax, xlabel="Mean Topic Proportion")
    save_show(fig, f"{prefix}_before_after.png")

    colors = np.where(
        comparison["delta"].gt(delta_threshold),
        TC.INC,
        np.where(
            comparison["delta"].lt(-delta_threshold),
            TC.DEC,
            TC.NEUTRAL,
        ),
    )
    fig, ax = plt.subplots(
        figsize=(9, max(4.5, len(comparison) * 0.65))
    )
    ax.barh(y, comparison["delta"], color=colors, alpha=0.88)
    ax.axvline(0, color=TC.TEXT_D, linewidth=1.2)
    offset = max(comparison["abs_delta"].max() * 0.03, 0.0002)
    for index, delta in enumerate(comparison["delta"]):
        ax.text(
            delta + (offset if delta >= 0 else -offset),
            index,
            f"Δ {delta:+.4f}",
            va="center",
            ha="left" if delta >= 0 else "right",
            fontsize=8.5,
            color=TC.TEXT_M,
        )
    ax.set_yticks(y)
    ax.set_yticklabels(comparison["label"])
    ax.set_title(
        f"Δ Change in {result.name} Topic Proportion",
        fontsize=13,
        fontweight="bold",
        color=TC.TEXT_D,
        pad=12,
    )
    style_axis(
        ax,
        xlabel="Proportion Change (Δ = After − Before)",
    )
    save_show(fig, f"{prefix}_delta.png")

In [ ]:
for sentiment, result in main_results.items():
    prefix = "main_pos" if sentiment == "positive" else "main_neg"
    plot_topic_comparison(result, prefix)

    display_table = result.comparison.copy()
    display_table["top_words"] = display_table["topic_num"].map(
        lambda topic_num: topic_label(
            result.topic_words,
            topic_num,
            n=10,
        )
    )
    display(
        display_table[
            [
                "topic_num",
                "top_words",
                "before",
                "after",
                "delta",
            ]
        ].sort_values("delta", ascending=False)
    )

In [ ]:
def plot_topic_keywords(
    result: LDAResult,
    topic_num: int,
    prefix: str,
    topn: int = 15,
) -> None:
    words = (
        result.topic_words.loc[
            result.topic_words["topic_num"].eq(topic_num)
        ]
        .nsmallest(topn, "rank")
        .sort_values("probability")
    )
    fig, ax = plt.subplots(figsize=(8, max(4, topn * 0.32)))
    ax.barh(
        words["word"],
        words["probability"],
        color=TC.NAVY,
        alpha=0.88,
    )
    ax.set_title(
        f"{result.name} T{topic_num}: Top Keywords",
        fontsize=13,
        fontweight="bold",
        color=TC.TEXT_D,
        pad=12,
    )
    style_axis(ax, xlabel="Term Probability P(word | topic)")
    save_show(fig, f"{prefix}_t{topic_num}_keywords.png")


for sentiment, result in main_results.items():
    prefix = "main_pos" if sentiment == "positive" else "main_neg"
    for topic_num in range(1, result.k + 1):
        plot_topic_keywords(result, topic_num, prefix)

## 9. 하위 LDA 대상 선택

LDA 토픽 번호는 재학습할 때 바뀔 수 있으므로 기존 T2·T6 번호를 하드코딩하지
않는다. 연구에서 다룬 세 의미 영역을 사전 정의한 anchor keyword와 상위 토픽
단어의 중첩으로 다시 찾는다.

- Positive product feedback
- Negative AI distrust
- Negative subscription and limits

선택 결과와 Agent 전후 변화량을 표로 저장하여 연구자가 대표 문서를 검토할 수
있게 한다.

In [ ]:
SUB_TOPIC_ANCHORS = {
    "positive_product_feedback": {
        "sentiment": "positive",
        "anchors": {
            "image", "feature", "version", "response", "picture",
            "limit", "user", "experience", "voice", "update",
        },
    },
    "negative_ai_distrust": {
        "sentiment": "negative",
        "anchors": {
            "memory", "fact", "truth", "people", "human",
            "company", "tool", "life", "world", "model",
        },
    },
    "negative_subscription_limits": {
        "sentiment": "negative",
        "anchors": {
            "subscription", "money", "limit", "hour", "month",
            "plan", "price", "payment", "waste", "time",
        },
    },
}


def select_topic_by_anchors(
    result: LDAResult,
    anchors: set[str],
    topn: int = 20,
) -> tuple[int, pd.DataFrame]:
    rows = []
    for topic_num in range(1, result.k + 1):
        words = set(
            result.topic_words.loc[
                result.topic_words["topic_num"].eq(topic_num)
                & result.topic_words["rank"].le(topn),
                "word",
            ]
        )
        overlap = sorted(words & anchors)
        delta = result.comparison.loc[
            result.comparison["topic_num"].eq(topic_num),
            "delta",
        ].iloc[0]
        rows.append(
            {
                "topic_num": topic_num,
                "anchor_overlap": len(overlap),
                "matched_words": ", ".join(overlap),
                "delta": delta,
            }
        )
    ranking = pd.DataFrame(rows).sort_values(
        ["anchor_overlap", "delta"],
        ascending=[False, False],
    )
    return int(ranking.iloc[0]["topic_num"]), ranking


selected_parent_topics = {}
selection_rows = []
for label, config in SUB_TOPIC_ANCHORS.items():
    result = main_results[config["sentiment"]]
    topic_num, ranking = select_topic_by_anchors(
        result,
        config["anchors"],
    )
    selected_parent_topics[label] = {
        "sentiment": config["sentiment"],
        "topic_num": topic_num,
    }
    best = ranking.iloc[0]
    selection_rows.append(
        {
            "label": label,
            "sentiment": config["sentiment"],
            "topic_num": topic_num,
            "anchor_overlap": int(best["anchor_overlap"]),
            "matched_words": best["matched_words"],
            "delta": best["delta"],
        }
    )

parent_topic_selection = pd.DataFrame(selection_rows)
parent_topic_selection.to_csv(
    TABLE_DIR / "sub_lda_parent_topic_selection.csv",
    index=False,
)
display(parent_topic_selection)

## 10. 하위 LDA 학습과 Coherence 선택

In [ ]:
def select_parent_documents(
    result: LDAResult,
    topic_num: int,
) -> pd.DataFrame:
    topic_column = f"topic_{topic_num}"
    selected_topics = result.doc_topics.loc[
        result.doc_topics["dominant_topic"].eq(topic_num)
        & result.doc_topics[topic_column].ge(
            PARENT_TOPIC_PROB_THRESHOLD
        )
    ][["document_index", topic_column]]

    selected = selected_topics.merge(
        result.documents.reset_index().rename(
            columns={"index": "document_index"}
        ),
        on="document_index",
        how="left",
    )
    counts = selected["period"].value_counts()
    if any(
        counts.get(period, 0) < MIN_SUB_DOCS_PER_PERIOD
        for period in ["before", "after"]
    ):
        raise ValueError(
            f"Parent T{topic_num}의 기간별 문서가 부족합니다: "
            f"{counts.to_dict()}"
        )
    return selected.reset_index(drop=True)


sub_results = {}
for label, config in selected_parent_topics.items():
    parent_result = main_results[config["sentiment"]]
    parent_documents = select_parent_documents(
        parent_result,
        config["topic_num"],
    )
    print(
        f"\n{label}: parent T{config['topic_num']}, "
        f"documents={len(parent_documents):,}"
    )
    sub_results[label] = train_lda(
        parent_documents,
        name=label.replace("_", " ").title(),
        k_range=SUB_K_RANGE,
        output_prefix=f"sub_{label}",
    )

## 11. 하위 LDA 전후 비중과 키워드 시각화

In [ ]:
for label, result in sub_results.items():
    plot_topic_comparison(
        result,
        prefix=f"sub_{label}",
        delta_threshold=0.01,
    )
    for topic_num in range(1, result.k + 1):
        plot_topic_keywords(
            result,
            topic_num,
            prefix=f"sub_{label}",
        )

    summary = result.comparison.copy()
    summary["top_words"] = summary["topic_num"].map(
        lambda topic_num: topic_label(
            result.topic_words,
            topic_num,
            n=10,
        )
    )
    display(
        summary.sort_values("abs_delta", ascending=False)[
            [
                "topic_num",
                "top_words",
                "before",
                "after",
                "delta",
            ]
        ]
    )

## 12. 대표 문서 추출

In [ ]:
def extract_representative_documents(
    result: LDAResult,
    n_each: int = 10,
) -> pd.DataFrame:
    source = result.documents.reset_index(drop=True).copy()
    source["_lda_document_index"] = np.arange(len(source))
    merged = result.doc_topics.merge(
        source[
            [
                "_lda_document_index",
                "review_id",
                "period",
                "score",
                "sentiment",
                "review",
            ]
        ],
        left_on=["document_index", "review_id", "period"],
        right_on=["_lda_document_index", "review_id", "period"],
        how="left",
    )

    rows = []
    for topic_num in range(1, result.k + 1):
        topic_column = f"topic_{topic_num}"
        top_words = topic_label(
            result.topic_words,
            topic_num,
            n=10,
        )
        for period in ["before", "after"]:
            top_documents = (
                merged.loc[merged["period"].eq(period)]
                .nlargest(n_each, topic_column)
            )
            for _, row in top_documents.iterrows():
                rows.append(
                    {
                        "model": result.name,
                        "topic_num": topic_num,
                        "top_words": top_words,
                        "period": period,
                        "score": row["score"],
                        "topic_probability": row[topic_column],
                        "review": row["review"],
                    }
                )
    return pd.DataFrame(rows)


representative_outputs = {}
for sentiment, result in main_results.items():
    representatives = extract_representative_documents(result)
    representatives.to_csv(
        REPRESENTATIVE_DIR / f"main_{sentiment}.csv",
        index=False,
        encoding="utf-8-sig",
    )
    representative_outputs[f"main_{sentiment}"] = representatives

for label, result in sub_results.items():
    representatives = extract_representative_documents(result)
    representatives.to_csv(
        REPRESENTATIVE_DIR / f"sub_{label}.csv",
        index=False,
        encoding="utf-8-sig",
    )
    representative_outputs[f"sub_{label}"] = representatives

display(representative_outputs["main_negative"].head(10))

## 13. 결과 요약

In [ ]:
sentiment_change = (
    sentiment_summary.pivot(
        index="sentiment",
        columns="period",
        values="share",
    )
    .reindex(["positive", "neutral", "negative"])
)
sentiment_change["change_pp"] = (
    sentiment_change["after"] - sentiment_change["before"]
) * 100

main_change_summary = []
for sentiment, result in main_results.items():
    largest = result.comparison.loc[
        result.comparison["abs_delta"].idxmax()
    ]
    main_change_summary.append(
        {
            "sentiment": sentiment,
            "topic_num": int(largest["topic_num"]),
            "top_words": topic_label(
                result.topic_words,
                int(largest["topic_num"]),
                n=10,
            ),
            "before": largest["before"],
            "after": largest["after"],
            "delta": largest["delta"],
        }
    )

main_change_summary = pd.DataFrame(main_change_summary)
main_change_summary.to_csv(
    TABLE_DIR / "main_change_summary.csv",
    index=False,
)

print("Sentiment change")
display(
    sentiment_change.style.format(
        {
            "before": "{:.2%}",
            "after": "{:.2%}",
            "change_pp": "{:+.2f} pp",
        }
    )
)
print("Largest topic movement by sentiment")
display(main_change_summary)

## 해석 지침

1. Coherence 최고값과 토픽 키워드를 함께 보고 k의 해석 가능성을 확인한다.
2. 상위 LDA의 Before/After 평균 토픽 비중과 Δ를 비교한다.
3. anchor keyword로 선택된 상위 토픽이 연구 개념과 일치하는지 대표 문서로 검토한다.
4. 하위 LDA에서도 Coherence 최고 k를 선택하고 세부 담론의 Δ를 확인한다.
5. 사용자 리뷰에서 관찰된 담론을 외부 사건의 객관적 사실로 단정하지 않는다.
6. 본 분석은 전후 비교이며 AI Agent 도입의 인과효과를 직접 증명하지 않는다.

추가 데이터를 반영하면 토픽 번호, 키워드, Coherence와 비중이 달라질 수 있다.
논문과 README에는 반드시 해당 실행의 데이터 종료일과 출력 폴더를 기록한다.

## 공개 전 체크리스트

- `RUN_MODE = "full"`, `USE_CACHE = False`로 전체 실행 성공 여부 확인
- 실행 환경과 패키지 버전을 `requirements.txt` 또는 `environment.yml`에 기록
- 원시 리뷰 CSV와 사용자 이름은 Git에서 제외
- 대표 문서 파일에는 사용자 식별자가 없음을 확인
- 집계 테이블과 그림만 `results` 또는 본 노트북의 출력 폴더에 공개
- 토픽 명칭은 키워드와 대표 문서를 검토한 뒤 README에 기재